In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
from typing import TypedDict, List, Any
from pydantic import BaseModel, Field
# from huggingface_hub import InferenceClient
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatResult, ChatGeneration
import numpy as np
import pandas as pd

In [40]:
application_train_exp = pd.read_csv("application_train_373139.csv")
bureau_balance_exp = pd.read_csv("bureau_balance_373139.csv")
bureau_exp = pd.read_csv("bureau_373139.csv")
credit_card_balance_exp = pd.read_csv("credit_card_balance_373139.csv")
installments_payments_exp = pd.read_csv("installments_payments_373139.csv")
POS_CASH_balance_exp = pd.read_csv("POS_CASH_balance_373139.csv")
previous_application_exp = pd.read_csv("previous_application_373139.csv")

In [41]:
final_dataset_exp = pd.read_csv("final_dataset_373139.csv")
final_dataset_exp

,AT_NAME_CONTRACT_TYPE,AT_CODE_GENDER,AT_FLAG_OWN_CAR,AT_FLAG_OWN_REALTY,AT_NAME_TYPE_SUITE,AT_NAME_INCOME_TYPE,AT_NAME_EDUCATION_TYPE,AT_NAME_FAMILY_STATUS,AT_NAME_HOUSING_TYPE,AT_OCCUPATION_TYPE,...,BB_STATUS_C_mean_mean,BB_STATUS_C_mean_max,BB_STATUS_Others_mean_mean,BB_STATUS_Others_mean_max,BB_STATUS_X_mean_mean,BB_STATUS_X_mean_max,BB_MONTHS_BALANCE_mean_mean,BB_MONTHS_BALANCE_mean_max,BB_MONTHS_BALANCE_max_mean,BB_MONTHS_BALANCE_max_max
0,0.0,1.0,1.0,1.0,6.0,6.0,4.0,3.0,1.0,0.0,...,0.46047,0.615385,0.0,0.0,0.066239,0.111111,-9.0,-2.5,0.0,0


In [42]:
table_definitions = {
    "application_train": [
        "Contains info about current loan and loan applicant at application time."
    ],
    "bureau": [
        "All client's previous credits provided by other financial institutions that were reported to Credit Bureau (for clients who have current loan).",
        "For every current loan, there are as many rows as number of credits the client had in Credit Bureau before the application date."
    ],
    "bureau_balance": [
        "Contains monthly balances of previous credits in Credit Bureau.",
        "Has one row for each month of history of every previous credit reported to Credit Bureau."
    ],
    "POS_CASH_balance": [
        "Contains monthly balance snapshots of previous POS (point of sales) and cash loans that the applicant had with Home Credit.",
        "Has one row for each month of history of every previous credit in Home Credit (consumer credit and cash loans) related to current loan."
    ],
    "credit_card_balance": [
        "Contains monthly balance snapshots of previous credit cards that the applicant has with Home Credit.",
        "Has one row for each month of history of every previous credit in Home Credit (consumer credit and cash loans) related to current loan."
    ],
    "previous_application": [
        "All previous applications for Home Credit loans of clients who have current loan.",
        "There is one row for each previous application related to current loan."
    ],
    "installments_payments": [
        "Repayment history for the previously disbursed credits in Home Credit related to current loan.",
        "There is a) one row for every payment that was made plus b) one row each for missed payment.",
        "One row is equivalent to one payment of one installment OR one installment corresponding to one payment of one previous Home Credit credit related to current loan."
    ] 
}

In [43]:
global_feature_importance = ['AT_EXT_SOURCE_2', 'AT_EXT_SOURCE_3', 'AT_DAYS_EMPLOYED',
       'AT_AMT_GOODS_PRICE', 'AT_AMT_CREDIT', 'AT_CODE_GENDER',
       'AT_DAYS_BIRTH', 'B_DAYS_CREDIT_max', 'AT_DAYS_ID_PUBLISH',
       'AT_AMT_ANNUITY', 'PC_CNT_INSTALMENT_FUTURE_mean_mean',
       'AT_NAME_EDUCATION_TYPE', 'PC_CNT_INSTALMENT_FUTURE_mean_max',
       'IP_AMT_PAYMENT_mean_max', 'B_AMT_CREDIT_SUM_max',
       'PA_CNT_PAYMENT_mean', 'PA_HOUR_APPR_PROCESS_START_mean',
       'IP_AMT_PAYMENT_mean_mean', 'AT_DAYS_REGISTRATION',
       'B_AMT_CREDIT_SUM_DEBT_max', 'AT_TOTALAREA_MODE',
       'PA_SELLERPLACE_AREA_mean', 'B_CREDIT_TYPE_Mortgage_mean',
       'PA_NAME_CLIENT_TYPE_New_mean', 'PC_SK_DPD_DEF_max_mean']

final_dataset_exp[global_feature_importance]

,AT_EXT_SOURCE_2,AT_EXT_SOURCE_3,AT_DAYS_EMPLOYED,AT_AMT_GOODS_PRICE,AT_AMT_CREDIT,AT_CODE_GENDER,AT_DAYS_BIRTH,B_DAYS_CREDIT_max,AT_DAYS_ID_PUBLISH,AT_AMT_ANNUITY,...,PA_CNT_PAYMENT_mean,PA_HOUR_APPR_PROCESS_START_mean,IP_AMT_PAYMENT_mean_mean,AT_DAYS_REGISTRATION,B_AMT_CREDIT_SUM_DEBT_max,AT_TOTALAREA_MODE,PA_SELLERPLACE_AREA_mean,B_CREDIT_TYPE_Mortgage_mean,PA_NAME_CLIENT_TYPE_New_mean,PC_SK_DPD_DEF_max_mean
0,0.413758,NaN,-2129,360000.0,360000.0,1.0,-9312,-178,-1665,21883.5,...,9.333333,12.0,7733.084107,-8884.0,0.0,NaN,24.666667,0.0,0.333333,0.0


In [44]:
def create_feature_dict(features_name: List[str], df_map: dict) -> dict:
    dict_ = {
        'application_train': [],
        'bureau_balance': [],
        'bureau': [],
        'credit_card_balance': [],
        'installments_payments': [],
        'POS_CASH_balance': [],
        'previous_application': []
    }
    detailed_dict_ = {}

    for feature in features_name:
        prefix = feature.split("_")[0]
        remain = feature.split("_")[1:]
        filtered_remain = [x for x in remain if x.isupper() or x.isnumeric()]
        feature = "_".join(filtered_remain)

        if prefix == "AT":
            dict_['application_train'].append(feature)
        elif prefix == "B":
            dict_['bureau'].append(feature)
        elif prefix == "BB":
            dict_['bureau_balance'].append(feature)
        elif prefix == "CC":
            dict_['credit_card_balance'].append(feature)
        elif prefix == "IP":
            dict_['installments_payments'].append(feature)
        elif prefix == "PC":
            dict_['POS_CASH_balance'].append(feature)
        elif prefix == "PA":
            dict_['previous_application'].append(feature)

    dict_ = {k: list(dict.fromkeys(v)) for k, v in dict_.items() if len(v) > 0}

    for name, features in dict_.items():          # features is a list
        detailed_dict_[name] = {}
        df = df_map[name]
        
        for feature in features:                  # iterate over each column name
            # .values → numpy array, .tolist() → Python list
            detailed_dict_[name][feature] = df[feature].values.tolist()

    return detailed_dict_

In [57]:
df_map = {
    "application_train": application_train_exp,
    "bureau": bureau_exp,
    "bureau_balance": bureau_balance_exp,
    "credit_card_balance": credit_card_balance_exp,
    "installments_payments": installments_payments_exp,
    "POS_CASH_balance": POS_CASH_balance_exp,
    "previous_application": previous_application_exp,
}

detailed_dict_ = create_feature_dict(global_feature_importance, df_map)
detailed_dict_

{'application_train': {'EXT_SOURCE_2': [0.4137577624682256],
  'EXT_SOURCE_3': [nan],
  'DAYS_EMPLOYED': [-2129],
  'AMT_GOODS_PRICE': [360000.0],
  'AMT_CREDIT': [360000.0],
  'CODE_GENDER': ['M'],
  'DAYS_BIRTH': [-9312],
  'DAYS_ID_PUBLISH': [-1665],
  'AMT_ANNUITY': [21883.5],
  'NAME_EDUCATION_TYPE': ['Secondary / secondary special'],
  'DAYS_REGISTRATION': [-8884.0],
  'TOTALAREA_MODE': [nan]},
 'bureau': {'DAYS_CREDIT': [-178, -520, -761, -790],
  'AMT_CREDIT_SUM': [0.0, 61490.475, 19488.6, 39105.0],
  'AMT_CREDIT_SUM_DEBT': [0.0, 0.0, 0.0, 0.0],
  'CREDIT_TYPE': ['Credit card',
   'Consumer credit',
   'Consumer credit',
   'Consumer credit']},
 'installments_payments': {'AMT_PAYMENT': [3171.195,
   14536.71,
   3171.195,
   14534.505,
   3168.675,
   2063.61,
   2.52,
   2063.61,
   11580.93,
   2063.61,
   2063.61,
   19236.87,
   14536.71,
   2063.61,
   14536.71,
   3171.195,
   3171.195]},
 'POS_CASH_balance': {'CNT_INSTALMENT_FUTURE': [11.0,
   1.0,
   3.0,
   12.0,
   0.

In [45]:
def create_feature_description(for_dict: dict, description_file_path: str) -> dict:
    description = pd.read_csv(description_file_path, encoding='latin1')
    desc_lookup = {}
    feature_descriptions = {}
    
    # Normalize table names if needed (optional but safer)
    table_name_map = {
        "application_train": "application_{train|test}",
        "bureau": "bureau",
        "bureau_balance": "bureau_balance",
        "credit_card_balance": "credit_card_balance",
        "installments_payments": "installments_payments",
        "POS_CASH_balance": "POS_CASH_balance",
        "previous_application": "previous_application",
    }
    
    for _, row in description.iterrows():
        table = row["Table"]
        feature = row["Row"]
        desc = row["Description"]
        special = row["Special"]
        desc_lookup[f"{table}_{feature}"] = f"{desc} (Special: {special})"
    
    for table, features in for_dict.items():
        feature_descriptions[table] = {}
        
        # Get the official table name used in the description file
        official_table = table_name_map.get(table)
        
        for feature, values in features.items():
            # Try exact match first
            desc = desc_lookup.get(f"{official_table}.csv_{feature}", 1)
            
            feature_descriptions[table][feature] = desc
    
    return feature_descriptions

In [14]:
feature_descriptions = create_feature_description(for_dict=detailed_dict_, description_file_path="/Users/thuyduc/Projects/AI/AI Sandbox/Services/CreditRiskAgent/data/raw/HomeCredit_columns_description.csv")
feature_descriptions

{'application_train': {'EXT_SOURCE_2': 'Normalized score from external data source (Special: normalized)',
  'EXT_SOURCE_3': 'Normalized score from external data source (Special: normalized)',
  'DAYS_EMPLOYED': 'How many days before the application the person started current employment (Special: time only relative to the application)',
  'AMT_GOODS_PRICE': 'For consumer loans it is the price of the goods for which the loan is given (Special: nan)',
  'AMT_CREDIT': 'Credit amount of the loan (Special: nan)',
  'CODE_GENDER': 'Gender of the client (Special: nan)',
  'DAYS_BIRTH': "Client's age in days at the time of application (Special: time only relative to the application)",
  'DAYS_ID_PUBLISH': 'How many days before the application did client change the identity document with which he applied for the loan (Special: time only relative to the application)',
  'AMT_ANNUITY': 'Loan annuity (Special: nan)',
  'NAME_EDUCATION_TYPE': 'Level of highest education the client achieved (Special

In [46]:
optimal_threshold_calibrated = 0.08644900452917655

output_from_ml_model_shap = {
    'baseline value': np.float64(0.0722),
    'probability for positive class': np.float64(0.0885),
    'late payment?': 'True',
    'top feature attribution': {
        'AT_CODE_GENDER': np.float64(0.0183),
        'AT_AMT_GOODS_PRICE': np.float64(0.0102),
        'B_CREDIT_ACTIVE_Closed_mean': np.float64(-0.0071),
        'B_AMT_CREDIT_SUM_DEBT_mean': np.float64(-0.0051),
        'AT_NAME_EDUCATION_TYPE': np.float64(0.0045),
        'AT_NAME_INCOME_TYPE': np.float64(0.0035),
        'IP_AMT_INSTALMENT_max_max': np.float64(-0.0026),
        'PA_PRODUCT_COMBINATION_Cash Street: middle_mean': np.float64(-0.0025),
        'PA_NAME_TYPE_SUITE_Unaccompanied_mean': np.float64(-0.0019),
        'PA_SELLERPLACE_AREA_max': np.float64(-0.0012)
    }
}

In [48]:
feature_attributions = output_from_ml_model_shap['top feature attribution'].keys()

detailed_dict_for_features_attributions = create_feature_dict(feature_attributions, df_map)
detailed_dict_for_features_attributions

{'application_train': {'CODE_GENDER': ['M'],
  'AMT_GOODS_PRICE': [360000.0],
  'NAME_EDUCATION_TYPE': ['Secondary / secondary special'],
  'NAME_INCOME_TYPE': ['Working']},
 'bureau': {'CREDIT_ACTIVE': ['Active', 'Closed', 'Closed', 'Closed'],
  'AMT_CREDIT_SUM_DEBT': [0.0, 0.0, 0.0, 0.0]},
 'installments_payments': {'AMT_INSTALMENT': [3171.195,
   14536.71,
   3171.195,
   14534.505,
   3171.195,
   2063.61,
   3171.195,
   2063.61,
   11580.93,
   2063.61,
   2063.61,
   19236.87,
   14536.71,
   2063.61,
   14536.71,
   3171.195,
   3171.195]},
 'previous_application': {'PRODUCT_COMBINATION': ['POS other with interest',
   'POS mobile with interest',
   'POS household with interest'],
  'NAME_TYPE_SUITE': ['Unaccompanied', 'Unaccompanied', 'Family'],
  'SELLERPLACE_AREA': [12, 32, 30]}}

In [111]:
class AgentState(TypedDict):
    customer_infomation: dict     # Thông tin khách hàng
    prob_positive: float          # Xác suất dự đoán của nhãn Positive
    baseline_value: float         # Giá trị baseline từ SHAP
    late_payment: str             # Có thanh toán trễ hay không (Boolean)
    top_feature_contribution: dict# Đóng góp của các đặc trưng quan trọng nhất (theo SHAP) vào xác suất dự đoán
    
    customer_summary: str         # Một bản tóm tắt thông tin khách hàng
    customer_credit_risk_explaination: str # Một bản báo cảo xác xuất khách hàng trễ hạn thanh toán trong đơn vay hiện tại, kèm theo lời giải thích

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
class CreditRiskAgent:
    def __init__(self, llm: ChatGoogleGenerativeAI):
        self.llm = llm

    def create_feature_description(self, for_dict:dict, description_file_path:str) -> dict:
        description = pd.read_csv(description_file_path, encoding='latin1')
        desc_lookup = {}
        feature_descriptions = {}
        
        # Normalize table names if needed (optional but safer)
        table_name_map = {
            "application_train": "application_{train|test}",
            "bureau": "bureau",
            "bureau_balance": "bureau_balance",
            "credit_card_balance": "credit_card_balance",
            "installments_payments": "installments_payments",
            "POS_CASH_balance": "POS_CASH_balance",
            "previous_application": "previous_application",
        }
        
        for _, row in description.iterrows():
            table = row["Table"]
            feature = row["Row"]
            desc = row["Description"]
            special = row["Special"]
            desc_lookup[f"{table}_{feature}"] = f"{desc} (Special: {special})"
        
        for table, features in for_dict.items():
            feature_descriptions[table] = {}
            
            # Get the official table name used in the description file
            official_table = table_name_map.get(table)
            
            for feature, values in features.items():
                # Try exact match first
                desc = desc_lookup.get(f"{official_table}.csv_{feature}", 1)
                
                feature_descriptions[table][feature] = desc
        
        return feature_descriptions
    
    def get_table_definitions(self, detailed_dict:dict):
        selected_tables = detailed_dict.keys()
        table_definitions_text = "\n".join(
            f"- {table_name}:\n" +
            "\n".join(f"    - {definition}" for definition in table_definitions[table_name])
            for table_name in selected_tables
        )

        return table_definitions_text
    
    def customer_summarize_node(self, state: AgentState):
        """
        Tạo ra một bản tóm tắt hồ sơ khách hàng, về các lịch sử giao dịch trong quá khứ, hành vi khách hàng, lý lịch của khách hàng,...
        """
        print("--- Creating customer summarisation report ---")
        customer_info = state["customer_infomation"] # output a dictionary

        table_definitions_text = self.get_table_definitions(customer_info)

        feature_descriptions = self.create_feature_description(for_dict=customer_info, description_file_path="/Users/thuyduc/Projects/AI/AI Sandbox/Services/CreditRiskAgent/data/raw/HomeCredit_columns_description.csv")
        
        sys_message = SystemMessage(
            content=f"""
            You are a professional Credit Risk Analyst Agent.

            Your task is to carefully read the customer's data (provided by user) and generate a clear, structured, and insightful summary of the customer.

            ### Input Data Structure
            You will receive a dictionary with the following format:
            {{
                "table_name": {{
                    "feature_name": [value1, value2, ...],
                    ...
                }},
                ...
            }}

            - Some features contain a single value (from `application_train`).
            - Other features contain a list of values (from related tables like `bureau`, `installments_payments`, `POS_CASH_balance`, etc.).

            ### Table Definitions
            {table_definitions_text}

            ### Feature Definitions
            {feature_descriptions}

            ### Instructions
            1. Analyze every feature in given dictionary.
            2. Use the table and feature definitions to correctly interpret the meaning of each value.
            3. Pay special attention to:
            - Missing values (NaN)
            - Negative values (especially days-related features → they represent days before application)
            - Multiple values in related tables (summarize patterns, averages, extremes, trends)
            4. Write a professional and comprehensive customer summary that covers all useful insights. Structure the summary into the following sections (use these exact headings):
                **1. Demographic & Personal Profile**
                - Age, gender, education level, and other personal characteristics.
                - Housing / registration related information (if available).

                **2. Employment & Income Situation**
                - Current employment duration and stability.
                - Any available income or annuity information.

                **3. Current Loan Application**
                - Loan amount, goods price, annuity, and other key application details.
                - External credit scores (EXT_SOURCE) if available.

                **4. Credit History Overview**
                - Number and types of previous credits (from bureau / previous applications).
                - Credit amounts, outstanding debt, and credit types.
                - How recent the previous credits are.

                **5. Payment Behavior & History**
                - Payment patterns from installments (whether the customer paid the full installment amount, underpaid, or overpaid).
                - Frequency and consistency of payments.
                - Any signs of late payments or difficulties (if available from DPD or similar features).

                **6. Ongoing Credit Obligations**
                - Remaining installments (CNT_INSTALMENT_FUTURE).
                - Current status of previous POS / cash loans or other active products.

                **7. Overall Behavioral Insights & Risk Signals**
                - Summarize the customer's overall credit behavior (responsible / risky / mixed).
                - Highlight any positive signals (good payment history, stable employment, etc.).
                - Highlight any concerning signals (high debt, missing data, irregular payments, short credit history, etc.).
                - Do NOT give a final approve/reject decision — only describe observations.

            ### Output Format
            Write in clear, professional English. Be specific and use numbers from the data whenever possible. Do not include any extra explanation or markdown code blocks.
            """
        )

        human_message = HumanMessage(
            content=f"""Here is the customer's detailed data: 
            {customer_info}
            Please generate a professional summary of this customer.""")
        
        messages = [
            sys_message,
            human_message
        ]
        
        result = self.llm.invoke(messages)
        
        return {"customer_summary": result.content}

    def create_feature_dict(self, features_name: List[str], df_map: dict) -> dict:
        dict_ = {
            'application_train': [],
            'bureau_balance': [],
            'bureau': [],
            'credit_card_balance': [],
            'installments_payments': [],
            'POS_CASH_balance': [],
            'previous_application': []
        }
        detailed_dict_ = {}

        for feature in features_name:
            prefix = feature.split("_")[0]
            remain = feature.split("_")[1:]
            filtered_remain = [x for x in remain if x.isupper() or x.isnumeric()]
            feature = "_".join(filtered_remain)

            if prefix == "AT":
                dict_['application_train'].append(feature)
            elif prefix == "B":
                dict_['bureau'].append(feature)
            elif prefix == "BB":
                dict_['bureau_balance'].append(feature)
            elif prefix == "CC":
                dict_['credit_card_balance'].append(feature)
            elif prefix == "IP":
                dict_['installments_payments'].append(feature)
            elif prefix == "PC":
                dict_['POS_CASH_balance'].append(feature)
            elif prefix == "PA":
                dict_['previous_application'].append(feature)

        dict_ = {k: list(dict.fromkeys(v)) for k, v in dict_.items() if len(v) > 0}

        for name, features in dict_.items():          # features is a list
            detailed_dict_[name] = {}
            df = df_map[name]
            
            for feature in features:                  # iterate over each column name
                # .values → numpy array, .tolist() → Python list
                detailed_dict_[name][feature] = df[feature].values.tolist()

        return detailed_dict_
    
    def customer_credit_risk_explaination_node(self, state: AgentState):
        """
        Tạo ra một bản báo cáo xác xuất khách hàng trễ hạn thanh toán trong đơn vay hiện tại từ mô hình ML, kèm theo lời giải thích tại sao khách hàng có xác suất trễ hạn thanh toán cao hoặc thấp, dựa trên các đặc trưng quan trọng nhất (theo SHAP).
        """
        print("--- Creating customer credit risk explaination report ---")
        df_name_map = {
            "AT": "application_train",
            "B": "bureau",
            "BB": "bureau_balance",
            "CC": "credit_card_balance",
            "IP": "installments_payments",
            "PC": "POS_CASH_balance",
            "PA": "previous_application",
        }
        
        prob_positive = state["prob_positive"]
        baseline_value = state["baseline_value"]
        late_payment = state["late_payment"]
        top_feature_contribution = state["top_feature_contribution"]
        
        dict_ = self.create_feature_dict(list(top_feature_contribution.keys()), df_map)
        
        final_dict = {
            table: {
                feature: {
                    "value": values
                }
                for feature, values in features.items()
            }
            for table, features in dict_.items()
        }

        for key, value in top_feature_contribution.items():
            prefix = key.split("_")[0]
            remain = key.split("_")[1:]
            filtered_remain = [x for x in remain if x.isupper() or x.isnumeric()]
            table_name = df_name_map.get(prefix)
            feature_name = "_".join(filtered_remain)
            final_dict[table_name][feature_name]["contribution"] = float(value)

        table_definitions_text = self.get_table_definitions(final_dict)
        
        feature_descriptions = self.create_feature_description(for_dict=final_dict, description_file_path="/Users/thuyduc/Projects/AI/AI Sandbox/Services/CreditRiskAgent/data/raw/HomeCredit_columns_description.csv")
        
        sys_message = SystemMessage(
            content=f"""
            You are a professional Credit Risk Analyst Agent specialized in explaining late payment risk predictions.
            Your task is to generate a clear, professional, and easy-to-understand report about the late payment risk of a specific customer, based on the machine learning model prediction and SHAP explanations.
            ### Available Information
            - **prob_positive**: The Machine Learning model's predicted probability that this customer will have a late payment on the current loan (value between 0 and 1).
            - **baseline_value**: The average predicted probability across all customers (SHAP baseline). This is the starting point before considering this customer's specific features.
            - **late_payment**: Boolean indicating whether the Machine Learning model predicts this customer will have a late payment (`True`) or not (`False`).
            - **final_dict**: Dictionary containing the top features that most influence the prediction.
                Structure:
                {{
                    "table_name": {{
                        "feature_name": {{
                            "value": [actual value(s) of the customer],
                            "contribution": shap_value  # positive = increases risk, negative = decreases risk
                        }},
                        ...
                    }},
                    ...
                }}
            - **feature_descriptions**: Definitions of the features.
            - **table_definitions**: Description of each data table.
            ### Instructions for the Report
            Write a professional report with the following structure (use these exact section headings):

            **1. Late Payment Risk Prediction**
            - Clearly state the predicted probability (`prob_positive`) as a percentage.
            - State whether the Machine Learning model predicts the customer will have a late payment or not (`late_payment`).
            - Compare the customer's probability with the baseline (`baseline_value`).  
            Example: "The customer's risk is X% higher/lower than the average customer."

            **2. Key Drivers of the Prediction**
            - Explain the most important features from `final_dict` that push the prediction up or down from the baseline.
            - For each important feature:
            - Mention the actual value of the customer.
            - Explain what the feature means (use `feature_descriptions`).
            - State whether its contribution **increases** or **decreases** the late payment risk.
            - Quantify the impact when useful (e.g., "This feature increases the risk by approximately 0.018").
            - Group the explanation by positive contributors (increase risk) and negative contributors (decrease risk) if it improves clarity.
            - Focus on the top features that have the largest absolute contribution.

            **3. Overall Risk Assessment**
            - Summarize the main reasons why this customer has higher or lower risk than average.
            - Highlight any notable patterns (e.g., education level, income type, credit history status, loan amount, etc.).
            - Keep the tone professional, objective, and neutral. Do **not** give a final approve/reject recommendation.

            ### Important Rules
            - Always use the actual numbers from `prob_positive`, `baseline_value`, and the `contribution` values.
            - Explain in clear business language (avoid pure technical jargon).
            - When a feature has multiple values (e.g. from bureau), summarize the pattern instead of listing every single value.
            - Do not invent information that is not provided.
            - Return **only** the report using the exact section headings above.  
            Do not add any introductory sentence, conclusion outside the structure, or markdown code blocks.
            """
        )
        
        human_message = HumanMessage(
            content=f"""
            Here is the information for this customer:

            - Predicted probability of late payment (prob_positive): {prob_positive}
            - Baseline probability (baseline_value): {baseline_value}
            - Machine Learning Model prediction (late_payment): {late_payment}

            Top contributing features (final_dict):
            {final_dict}

            Feature definitions:
            {feature_descriptions}

            Table definitions:
            {table_definitions_text}

            Please generate the late payment risk report.
        """
        )

        messages = [
            sys_message,
            human_message
        ]
        result = self.llm.invoke(messages)
        
        return {'customer_credit_risk_explaination': result.content}

In [126]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # Load environment variables from .env file
GEMINI_API = os.getenv('GEMINI_API')
class GeminiModel:
    def __init__(self, api_key: str, model_name: str = "gemini-3-flash-preview"):
        self.api_key = api_key
        self.model_name = model_name
    
    def init_gemini_model(self):
        gemini = ChatGoogleGenerativeAI(
            model=self.model_name,
            max_output_tokens=50000,
            temperature=1.0,
            top_p=0.8,
            google_api_key=self.api_key # Use your actual API key here
        )
        return gemini

In [127]:
gemini_llm = GeminiModel(GEMINI_API, model_name="gemini-3-flash-preview").init_gemini_model()
credit_risk_agent = CreditRiskAgent(gemini_llm)

In [128]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AgentState)

workflow.add_node("customer_summarisation_node", credit_risk_agent.customer_credit_risk_explaination_node)
workflow.add_edge(START, "customer_summarisation_node")
workflow.add_edge("customer_summarisation_node", END)

app = workflow.compile()

In [129]:
initial_state = {
    "customer_infomation": detailed_dict_,
    "prob_positive": output_from_ml_model_shap['probability for positive class'],
    "baseline_value": output_from_ml_model_shap['baseline value'],
    "late_payment": output_from_ml_model_shap['late payment?'],
    "top_feature_contribution": output_from_ml_model_shap['top feature attribution']
}
result, feature_descriptions = app.invoke(initial_state)

--- Creating customer credit risk explaination report ---


InvalidUpdateError: Expected dict, got ({'customer_credit_risk_explaination': [{'type': 'text', 'text': '**1. Late Payment Risk Prediction**\n- The predicted probability of a late payment for this customer is 8.85%.\n- The Machine Learning model predicts that this customer will have a late payment on the current loan.\n- The customer\'s risk is 1.63% higher than the average customer baseline, which stands at 7.22%.\n\n**2. Key Drivers of the Prediction**\n\n**Factors Increasing Late Payment Risk:**\n- **Gender (CODE_GENDER):** The client is male. This demographic factor is the strongest driver in this profile, increasing the predicted risk by approximately 0.0183.\n- **Goods Price (AMT_GOODS_PRICE):** The price of the goods for which the loan is given is 360,000.00. This specific loan amount for the consumer goods increases the risk by 0.0102.\n- **Education Level (NAME_EDUCATION_TYPE):** The customer has a "Secondary / secondary special" education level. This contributes to an increas

In [124]:
print(result["customer_credit_risk_explaination"][0]['text'])

**1. Late Payment Risk Prediction**
- The predicted probability of a late payment for this customer is 8.85%.
- Based on this probability, the Machine Learning model predicts that the customer will have a late payment.
- The customer's risk is 1.63 percentage points higher than the baseline average of 7.22%.

**2. Key Drivers of the Prediction**

**Factors Increasing Late Payment Risk:**
- **Gender (CODE_GENDER):** The client is male ('M'), which is the strongest driver in this profile, increasing the late payment risk by approximately 0.0183.
- **Goods Price (AMT_GOODS_PRICE):** The price of the goods for which the loan is given is 360,000.0. This specific loan amount contributes to an increase in risk of 0.0102.
- **Education Level (NAME_EDUCATION_TYPE):** The client’s highest education level is "Secondary / secondary special," which increases the risk by 0.0045.
- **Income Type (NAME_INCOME_TYPE):** The client’s income type is categorized as "Working," which adds 0.0035 to the risk 

In [28]:
with open("workflow_graph.png", "wb") as f:
    f.write(app.get_graph().draw_mermaid_png())